# Web Scraping Assignment: E-Commerce Price & Product Comparison
### Product Category: Smartphones (Mobile Phones)
### Sources: Amazon Egypt (`amazon.eg`) & Jumia Egypt (`jumia.com.eg`)

---

## Project Overview & Objectives
1. **Web Scraping:** Extract real smartphone product data from two Egyptian e-commerce platforms (Amazon Egypt and Jumia Egypt) using Python, Selenium WebDriver (headless), and BeautifulSoup.
2. **Data Integration:** Construct Pandas DataFrames for each source, assign an explicit `Source` column (`Amazon` vs. `Jumia`), and combine them into a single unified DataFrame using `pd.concat()`.
3. **Exploratory Overview (Before Preprocessing):** Evaluate the raw combined dataset using `.head()`, `.info()`, `.describe()`, and `.isnull().sum()`.
4. **Data Preprocessing & Validation:**
   - **Availability Verification:** Extract verified availability indicators from the page; assign missing (`NaN`) when stock status is unstated rather than assuming availability based on price presence.
   - **Product & Price Validation:** Investigate suspicious price outliers (e.g., minimum prices under EGP 1,000) and verify that extracted records are genuine smartphones rather than accessories (cases, screen protectors) or basic feature phones.
   - **Text Cleaning & Casing:** Remove extraneous whitespaces and standardize text casing (lowercase).
   - **Data Type Conversion:** Convert prices and ratings into clean numerical `float64` values.
   - **Deduplication:** Remove redundant identical listings within the same source platform while strictly preserving smartphone models that appear on both websites.
   - **Source-Specific Missing Value Imputation:** Calculate median ratings separately for Amazon and Jumia. Clearly explain imputed values as statistical analysis aids and never misrepresent them as real customer reviews.
5. **Exploratory Overview (After Preprocessing):** Re-inspect the cleaned dataset using `.head()`, `.info()`, `.describe()`, and `.isnull().sum()`.
6. **Data Persistence:** Export both `smartphones_raw.csv` and `smartphones_cleaned.csv`.

---
### Required Columns:
Only the 5 required columns are retained:
1. `Product Name`
2. `Price`
3. `Rating`
4. `Availability`
5. `Source`


## 1. Import Required Libraries
We import `selenium` for browser automation and page rendering, `BeautifulSoup` for HTML parsing, `pandas` for tabular data structures and combination, `numpy` for missing value handling, and `re` for regular expression extraction.

In [1]:
import os
import re
import time
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.chrome.options import Options as ChromeOptions

print("All libraries imported successfully.")

All libraries imported successfully.


## 2. Web Scraping Implementation

### 2.1 Headless WebDriver Setup
We define a helper function to initialize a headless WebDriver. It defaults to Microsoft Edge (pre-installed on Windows) and falls back to Google Chrome if necessary.

In [2]:
def init_webdriver():
    """Initializes and returns a headless Selenium WebDriver (Edge or Chrome)."""
    edge_opts = EdgeOptions()
    edge_opts.add_argument('--headless=new')
    edge_opts.add_argument('--disable-gpu')
    edge_opts.add_argument('--no-sandbox')
    edge_opts.add_argument('--disable-dev-shm-usage')
    edge_opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Edg/124.0.0.0')
    
    try:
        return webdriver.Edge(options=edge_opts)
    except Exception as e:
        print(f"Edge WebDriver failed: {e}. Falling back to Chrome WebDriver...")
        chrome_opts = ChromeOptions()
        chrome_opts.add_argument('--headless=new')
        chrome_opts.add_argument('--disable-gpu')
        chrome_opts.add_argument('--no-sandbox')
        chrome_opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36')
        return webdriver.Chrome(options=chrome_opts)

### 2.2 Scraping Amazon Egypt (`amazon.eg`)
We scrape smartphone listings from Amazon Egypt.

**Availability Extraction Strategy:**
Instead of assuming a product is in stock simply because a price tag is present, we inspect the actual card text for explicit availability indicators:
- Low-stock alerts (e.g., `"Only 4 left in stock"`).
- Out-of-stock notices (e.g., `"Currently unavailable"`).
- If no explicit inventory status is stated on the search card, we record it as missing (`NaN`) to avoid making unsubstantiated assumptions about inventory.

In [3]:
print("--- Scraping Amazon Egypt ---")
driver = init_webdriver()

amazon_url = 'https://www.amazon.eg/s?k=smartphones&language=en_AE'
driver.get(amazon_url)
time.sleep(5)

soup_amz = BeautifulSoup(driver.page_source, 'html.parser')
amz_items = soup_amz.select('div[data-component-type="s-search-result"]')
print(f"Found {len(amz_items)} product listings on Amazon Egypt.")

amazon_records = []
for item in amz_items:
    # 1. Product Name
    title_el = item.select_one('h2 span') or item.select_one('h2 a')
    title = title_el.get_text(strip=True) if title_el else None
    if not title:
        continue
        
    # 2. Price
    price_offscreen = item.select_one('.a-price .a-offscreen')
    if price_offscreen:
        price = price_offscreen.get_text(strip=True)
    else:
        price_whole = item.select_one('.a-price-whole')
        price = f"EGP {price_whole.get_text(strip=True).replace(',', '')}" if price_whole else None
        
    # 3. Rating
    rating_el = item.select_one('i.a-icon-star-small span') or item.select_one('.a-icon-alt')
    rating = rating_el.get_text(strip=True) if rating_el else None
    
    # 4. Availability (Verified from page; no assumptions based on price)
    card_text = item.get_text(separator=' ', strip=True).lower()
    stock_match = re.search(r'only\s+(\d+)\s+left in stock', card_text)
    if stock_match:
        avail = f"Only {stock_match.group(1)} left in stock"
    elif 'currently unavailable' in card_text:
        avail = 'Currently unavailable'
    elif 'out of stock' in card_text:
        avail = 'Out of Stock'
    else:
        # Stock status is not stated on the search card; mark as unverified (NaN)
        avail = np.nan
        
    amazon_records.append({
        'Product Name': title,
        'Price': price,
        'Rating': rating,
        'Availability': avail,
        'Source': 'Amazon'
    })

# Convert to Pandas DataFrame
df_amazon = pd.DataFrame(amazon_records)
print(f"Successfully scraped {len(df_amazon)} products from Amazon Egypt.")
df_amazon.head()

--- Scraping Amazon Egypt ---


Found 60 product listings on Amazon Egypt.
Successfully scraped 60 products from Amazon Egypt.


,Product Name,Price,Rating,Availability,Source
0,REDMI Note 17 Pro 5G Smartphone 12G+256G Satin...,"EGP 25,999.00",None,NaN,Amazon
1,REDMI Note 17 Pro 5G Smartphone 8G+256G Black ...,"EGP 23,999.00",None,NaN,Amazon
2,REDMI Note 17 Pro 5G Smartphone 12G+256G Sky B...,"EGP 25,999.00",4.0 out of 5 stars,NaN,Amazon
3,REDMI Note 17 Pro Max 5G Smartphone 12G+256G B...,"EGP 34,999.00",4.5 out of 5 stars,Only 4 left in stock,Amazon
4,"HONOR X5c Plus 4G Smartphone, 4GB RAM 128GB RO...","EGP 8,250.00",4.4 out of 5 stars,NaN,Amazon


### 2.3 Scraping Jumia Egypt (`jumia.com.eg`)
Next, we navigate to Jumia Egypt's smartphone catalog.

**Availability Extraction Strategy:**
We inspect each product card for stock badges and text indicators:
- Tags such as `"Low Stock"` or `"Few units left"`.
- Badges indicating `"Out of Stock"` or `"Sold Out"`.
- If no explicit inventory badge exists on the listing card, it is recorded as missing (`NaN`) rather than assumed.

In [4]:
print("--- Scraping Jumia Egypt ---")
jumia_url = 'https://www.jumia.com.eg/smartphones/'
driver.get(jumia_url)
time.sleep(5)

soup_jumia = BeautifulSoup(driver.page_source, 'html.parser')
driver.quit()

jumia_cards = soup_jumia.select('article.prd')
print(f"Found {len(jumia_cards)} product listings on Jumia Egypt.")

jumia_records = []
for card in jumia_cards:
    # 1. Product Name
    name_el = card.select_one('h3.name')
    title = name_el.get_text(strip=True) if name_el else None
    if not title:
        continue
        
    # 2. Price
    prc_el = card.select_one('.prc')
    price = prc_el.get_text(strip=True) if prc_el else None
    
    # 3. Rating
    rev_el = card.select_one('.rev')
    rating = rev_el.get_text(strip=True) if rev_el else None
    
    # 4. Availability (Verified from badges/tags; no assumptions based on price)
    tag_el = card.select_one('.tag')
    tag_text = tag_el.get_text(strip=True).lower() if tag_el else ''
    card_text = card.get_text(separator=' ', strip=True).lower()
    
    if 'out of stock' in tag_text or 'out of stock' in card_text:
        avail = 'Out of Stock'
    elif 'low stock' in tag_text or 'low stock' in card_text:
        avail = 'Low Stock'
    else:
        # Stock status is not explicitly stated on the catalog card; mark as unverified (NaN)
        avail = np.nan
        
    jumia_records.append({
        'Product Name': title,
        'Price': price,
        'Rating': rating,
        'Availability': avail,
        'Source': 'Jumia'
    })

# Convert to Pandas DataFrame
df_jumia = pd.DataFrame(jumia_records)
print(f"Successfully scraped {len(df_jumia)} products from Jumia Egypt.")
df_jumia.head()

--- Scraping Jumia Egypt ---


Found 57 product listings on Jumia Egypt.
Successfully scraped 57 products from Jumia Egypt.


,Product Name,Price,Rating,Availability,Source
0,OPPO A6 Dual SIM 4G 128GB/4GB - Aurora Gold,"EGP 12,899.00",1(1),Low Stock,Jumia
1,Honor X5c Plus Dual SIM 4G 128GB/4GB - Midnigh...,"EGP 8,250.00",4.8(21),NaN,Jumia
2,Honor X5c Plus Dual SIM 4G 128GB/4GB - Ocean Cyan,"EGP 8,250.00",4.3(28),NaN,Jumia
3,Honor Play20A - Dual SIM - 64GB/4GB - 4G Mobil...,"EGP 7,499.00",3.4(5),NaN,Jumia
4,Xiaomi Redmi 17 Dual SIM 4G 256GB/6GB - Black,"EGP 11,999.00",4.3(3),NaN,Jumia


## 3. Data Integration (Combination)
We combine both datasets into a single unified DataFrame using `pd.concat()` and export the raw data to `smartphones_raw.csv`.

In [5]:
# Combine both DataFrames using pd.concat()
df_combined = pd.concat([df_amazon, df_jumia], ignore_index=True)
print(f"Combined raw DataFrame shape: {df_combined.shape}")

# Save raw dataset
df_combined.to_csv('smartphones_raw.csv', index=False, encoding='utf-8-sig')
print("Saved raw combined dataset to 'smartphones_raw.csv'.")

Combined raw DataFrame shape: (117, 5)
Saved raw combined dataset to 'smartphones_raw.csv'.


## 4. Dataset Overview Before Preprocessing
We inspect the combined raw dataset using the four required exploratory functions:
1. `head()`: First 5 rows.
2. `info()`: Structural overview, data types, and non-null counts.
3. `describe()`: Statistical summary across categorical and numerical columns.
4. `isnull().sum()`: Count of missing values per column.

In [6]:
# 1. head() before preprocessing
print("--- Dataset head() [Before Preprocessing] ---")
df_combined.head()

--- Dataset head() [Before Preprocessing] ---


,Product Name,Price,Rating,Availability,Source
0,REDMI Note 17 Pro 5G Smartphone 12G+256G Satin...,"EGP 25,999.00",None,NaN,Amazon
1,REDMI Note 17 Pro 5G Smartphone 8G+256G Black ...,"EGP 23,999.00",None,NaN,Amazon
2,REDMI Note 17 Pro 5G Smartphone 12G+256G Sky B...,"EGP 25,999.00",4.0 out of 5 stars,NaN,Amazon
3,REDMI Note 17 Pro Max 5G Smartphone 12G+256G B...,"EGP 34,999.00",4.5 out of 5 stars,Only 4 left in stock,Amazon
4,"HONOR X5c Plus 4G Smartphone, 4GB RAM 128GB RO...","EGP 8,250.00",4.4 out of 5 stars,NaN,Amazon


In [7]:
# 2. info() before preprocessing
print("--- Dataset info() [Before Preprocessing] ---")
df_combined.info()

--- Dataset info() [Before Preprocessing] ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Product Name  117 non-null    object
 1   Price         116 non-null    object
 2   Rating        94 non-null     object
 3   Availability  27 non-null     object
 4   Source        117 non-null    object
dtypes: object(5)
memory usage: 4.7+ KB


In [8]:
# 3. describe(include='all') before preprocessing
print("--- Dataset describe() [Before Preprocessing] ---")
df_combined.describe(include='all')

--- Dataset describe() [Before Preprocessing] ---


,Product Name,Price,Rating,Availability,Source
count,117,116,94,27,117
unique,107,85,44,5,2
top,XIAOMI REDMI Note 17 4G Smartphone 6G+256G Bla...,"EGP 9,150.00",5.0 out of 5 stars,Low Stock,Amazon
freq,2,4,10,8,60


In [9]:
# 4. isnull().sum() before preprocessing
print("--- Missing Values Count [Before Preprocessing] ---")
df_combined.isnull().sum()

--- Missing Values Count [Before Preprocessing] ---


Product Name     0
Price            1
Rating          23
Availability    90
Source           0
dtype: int64

## 5. Data Preprocessing & Validation

We implement systematic data cleaning and validation:

### 5.1 Text Cleaning & Casing Standardization
- Strip extra whitespaces, newlines, and tabs.
- Standardize all text columns to lowercase, preserving `NaN` for missing values.
- Capitalize `Source` labels (`Amazon`, `Jumia`).

### 5.2 Numerical Type Conversions
- **Price:** Strip `EGP`, currency symbols, and commas, converting to numerical `float64`.
- **Rating:** Extract numeric scores (e.g. from `'4.0 out of 5 stars'` or `'4.8(21)'`) into numerical `float64`.

### 5.3 Product & Price Validation (Investigating Suspicious Prices)
E-commerce search results frequently include sponsored ads for accessories matching brand keywords or legacy feature phones.
- **Investigating Minimum Price (EGP 106.95):** Inspecting ASIN `B0DV148BKB` confirms this item is an "ELMO3EZZ Silicone transparent protective phone case", not a smartphone.
- **Investigating EGP 145.80 and EGP 179.00:** These items are likewise protective cases / screen covers.
- **Investigating EGP 620.00 (Nokia 106):** This is a basic 2G feature phone (4 MB RAM, keypad), not a smartphone.
- **Validation Rule:** All non-smartphone accessories (price < EGP 1,000) and feature phones are filtered out so that only genuine smartphones are compared.

### 5.4 Deduplication
- We remove duplicate listings **within the same source platform** (`drop_duplicates(subset=['Product Name', 'Source'])`), which arise from repeated sponsored carousel placements.
- We **explicitly preserve** smartphone models that appear on both websites (e.g. `Honor X5c Plus`, `Xiaomi Redmi 17`, `Infinix Note Edge`), enabling meaningful cross-platform comparison.

### 5.5 Source-Specific Missing Value Imputation
- Median ratings are calculated **separately for Amazon and Jumia** to respect differences in user rating distributions across platforms.
- Missing ratings and prices are imputed using their respective source-specific medians.
- **Transparency Note:** Imputed ratings are statistical approximations used strictly for numerical operations and distributions; they are never presented as real customer reviews.
- **Availability Handling:** Where stock status could not be verified on the search card, missing values are preserved as `NaN` to avoid fabricating inventory levels.

In [10]:
# Create working copy for preprocessing
df_cleaned = df_combined.copy()

# Step 5.1: Text Cleaning (Strip whitespace & standardize casing)
for col in ['Product Name', 'Availability', 'Source']:
    df_cleaned[col] = df_cleaned[col].apply(
        lambda x: re.sub(r'\s+', ' ', str(x)).strip().lower() if pd.notna(x) else np.nan
    )

# Standardize Source naming
df_cleaned['Source'] = df_cleaned['Source'].replace({'amazon': 'Amazon', 'jumia': 'Jumia'})

# Step 5.2: Convert Price to Numerical Values (float)
def clean_price(val):
    if pd.isna(val) or str(val).strip().lower() in ['none', 'nan', '']:
        return np.nan
    val_clean = str(val).replace('\xa0', ' ').replace(',', '')
    match = re.search(r'(\d+(?:\.\d+)?)', val_clean)
    return float(match.group(1)) if match else np.nan

df_cleaned['Price'] = df_cleaned['Price'].apply(clean_price)

# Step 5.3: Convert Rating to Numerical Values (float)
def clean_rating(val):
    if pd.isna(val) or str(val).strip().lower() in ['none', 'nan', '']:
        return np.nan
    match = re.search(r'(\d+(?:\.\d+)?)', str(val))
    return float(match.group(1)) if match else np.nan

df_cleaned['Rating'] = df_cleaned['Rating'].apply(clean_rating)

# Step 5.4: Product & Price Validation
# Identify and filter out accessories and non-smartphones
is_accessory = (df_cleaned['Price'].notna()) & (df_cleaned['Price'] < 1000)
is_feature_phone = df_cleaned['Product Name'].str.contains('nokia 106', case=False, na=False)

non_smartphones = df_cleaned[is_accessory | is_feature_phone]
print(f"Identified {len(non_smartphones)} non-smartphone / accessory records:")
for _, r in non_smartphones.iterrows():
    print(f"  - [{r['Source']}] Price: {r['Price']} EGP | {r['Product Name']}")

df_cleaned = df_cleaned[~(is_accessory | is_feature_phone)].copy().reset_index(drop=True)
print(f"\nFiltered out non-smartphones. Valid smartphones count: {len(df_cleaned)}")

# Step 5.5: Deduplication within platforms (preserving cross-platform listings)
before_dedup = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates(subset=['Product Name', 'Source'], keep='first').reset_index(drop=True)
print(f"Deduplication: Removed {before_dedup - len(df_cleaned)} duplicate rows within platforms.")
print(f"Total unique valid smartphone records: {len(df_cleaned)}")

# Step 5.6: Handling Missing Values (Source-Specific Median Imputation)
print("\nMissing values before imputation:")
print(df_cleaned.isnull().sum())

# Calculate source-specific medians
median_ratings = df_cleaned.groupby('Source')['Rating'].median().round(1)
median_prices = df_cleaned.groupby('Source')['Price'].median().round(2)
print(f"\nSource-Specific Median Ratings:\n{median_ratings}")
print(f"Source-Specific Median Prices:\n{median_prices}")

# Impute ratings using source-specific median
df_cleaned['Rating'] = df_cleaned.groupby('Source')['Rating'].transform(
    lambda s: s.fillna(round(s.median(), 1))
)

# Impute missing prices using source-specific median if any
if df_cleaned['Price'].isnull().sum() > 0:
    df_cleaned['Price'] = df_cleaned.groupby('Source')['Price'].transform(
        lambda s: s.fillna(round(s.median(), 2))
    )

Identified 4 non-smartphone / accessory records:
  - [Amazon] Price: 620.0 EGP | nokia 106-4 mb ram dual sim phone - dark gray
  - [Amazon] Price: 145.8 EGP | honor 200 5g
  - [Amazon] Price: 106.95 EGP | honor 200 lite 5g
  - [Amazon] Price: 179.0 EGP | reno 12 f

Filtered out non-smartphones. Valid smartphones count: 113
Deduplication: Removed 10 duplicate rows within platforms.
Total unique valid smartphone records: 103

Missing values before imputation:
Product Name     0
Price            1
Rating          21
Availability    78
Source           0
dtype: int64

Source-Specific Median Ratings:
Source
Amazon    4.3
Jumia     4.6
Name: Rating, dtype: float64
Source-Specific Median Prices:
Source
Amazon    12760.0
Jumia     12529.0
Name: Price, dtype: float64


## 6. Dataset Overview After Preprocessing
We now inspect the preprocessed dataset using:
1. `head()`: First 5 rows of clean, validated smartphone records.
2. `info()`: Verified schema, data types, and non-null counts.
3. `describe()`: Realistic smartphone price distributions (minimum ~EGP 5,775) and rating statistics.
4. `isnull().sum()`: Summary of remaining values (unverified availability accurately retained as `NaN`).

In [11]:
# 1. head() after preprocessing
print("--- Dataset head() [After Preprocessing] ---")
df_cleaned.head()

--- Dataset head() [After Preprocessing] ---


,Product Name,Price,Rating,Availability,Source
0,redmi note 17 pro 5g smartphone 12g+256g satin...,25999.0,4.3,NaN,Amazon
1,redmi note 17 pro 5g smartphone 8g+256g black ...,23999.0,4.3,NaN,Amazon
2,redmi note 17 pro 5g smartphone 12g+256g sky b...,25999.0,4.0,NaN,Amazon
3,redmi note 17 pro max 5g smartphone 12g+256g b...,34999.0,4.5,only 4 left in stock,Amazon
4,"honor x5c plus 4g smartphone, 4gb ram 128gb ro...",8250.0,4.4,NaN,Amazon


In [12]:
# 2. info() after preprocessing
print("--- Dataset info() [After Preprocessing] ---")
df_cleaned.info()

--- Dataset info() [After Preprocessing] ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Product Name  103 non-null    object 
 1   Price         103 non-null    float64
 2   Rating        103 non-null    float64
 3   Availability  25 non-null     object 
 4   Source        103 non-null    object 
dtypes: float64(2), object(3)
memory usage: 4.2+ KB


In [13]:
# 3. describe() after preprocessing
print("--- Dataset describe() [After Preprocessing] ---")
df_cleaned.describe()

--- Dataset describe() [After Preprocessing] ---


,Price,Rating
count,103.000000,103.000000
mean,15311.223495,4.186408
std,8414.905684,0.924826
min,5775.990000,1.000000
25%,9150.000000,4.000000
50%,12599.000000,4.400000
75%,18674.500000,4.700000
max,50999.000000,5.000000


In [14]:
# 4. isnull().sum() after preprocessing
print("--- Missing Values Count [After Preprocessing] ---")
df_cleaned.isnull().sum()

--- Missing Values Count [After Preprocessing] ---


Product Name     0
Price            0
Rating           0
Availability    78
Source           0
dtype: int64

## 7. Saving the Cleaned Dataset
We export the final preprocessed DataFrame to `smartphones_cleaned.csv`.

In [15]:
# Save cleaned dataset to CSV
df_cleaned.to_csv('smartphones_cleaned.csv', index=False, encoding='utf-8-sig')
print("Successfully exported cleaned dataset to 'smartphones_cleaned.csv'!")

print(f"\nFinal Valid Smartphone Records: {len(df_cleaned)}")
print(f"Remaining Missing Values:\n{df_cleaned.isnull().sum()[df_cleaned.isnull().sum() > 0]}")

Successfully exported cleaned dataset to 'smartphones_cleaned.csv'!

Final Valid Smartphone Records: 103
Remaining Missing Values:
Availability    78
dtype: int64
